In [ ]:
!pip install pygrog[dev]


# Utils Tour: Coil Compression and NLINV

This example introduces two utility routines from :mod:`pygrog.utils`:

1. **Coil compression** (:func:`~pygrog.utils.coil_compress`) — reduces the
   number of receiver coils via PCA to speed up downstream processing without
   significant SNR loss.
2. **NLINV coil calibration** (:func:`~pygrog.utils.nlinv_calib`) — estimates
   coil sensitivity maps from undersampled k-space data using the
   nonlinear-inverse (NLINV) algorithm.

All data use a **BrainWeb T1-weighted** phantom: a multi-coil dataset is
constructed by multiplying it with synthetic coil sensitivity maps.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from brainweb_dl import get_mri

## BrainWeb multi-coil dataset

We load a BrainWeb T1-weighted phantom and multiply it by synthetic coil
sensitivity maps (smooth Gaussian weighting centred at each coil position).



In [ ]:
def _synthetic_smaps(shape, n_coils=4):
    """Synthetic sensitivity maps (Gaussian envelope + linear phase)."""
    ny, nx = shape
    yy, xx = np.mgrid[-1 : 1 : ny * 1j, -1 : 1 : nx * 1j]
    smaps = []
    for angle in np.linspace(0.0, 2.0 * np.pi, n_coils, endpoint=False):
        cx, cy = np.cos(angle), np.sin(angle)
        gauss = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * 0.45**2))
        phase = np.exp(1j * (cx * xx + cy * yy))
        smaps.append(gauss * phase)
    smaps = np.asarray(smaps, dtype=np.complex64)
    smaps /= np.sqrt((np.abs(smaps) ** 2).sum(0, keepdims=True)) + 1e-12
    return smaps


image = get_mri(0, "T1")
image = np.flip(image, axis=(0, 2))[90].astype(np.float32)
image /= image.max() + 1e-8
image_shape = image.shape
n_coils = 16
smaps_gt = _synthetic_smaps(image_shape, n_coils=n_coils)

# Coil images: (n_coils, ny, nx)
coil_images = smaps_gt * image[np.newaxis].astype(np.complex64)

# Cartesian k-space: (n_coils, ny, nx)
kspace_full = np.fft.fftshift(
    np.fft.fft2(np.fft.ifftshift(coil_images, axes=(-2, -1))), axes=(-2, -1)
)

print(f"Coil images shape: {coil_images.shape}")
print(f"K-space shape    : {kspace_full.shape}")

In [ ]:
fig, axes = plt.subplots(2, n_coils // 2, figsize=(11, 4))
axes = axes.flatten()
for i in range(n_coils):
    axes[i].imshow(np.abs(coil_images[i]), cmap="gray", origin="lower")
    axes[i].set_title(f"Coil {i + 1}")
    axes[i].axis("off")
plt.suptitle("Synthetic multi-coil images (16 coils)")
plt.tight_layout()
plt.show()

## Coil compression

:func:`~pygrog.utils.coil_compress` reduces the number of virtual coils
using PCA on the k-space data.  It returns the compressed k-space and the
compression matrix ``W`` of shape ``(n_virtual, n_coils)``.

The argument ``n_coils`` accepts:

* an **integer** — exact number of virtual coils to keep, or
* a **float** in (0, 1] — energy fraction to retain.



In [ ]:
from pygrog.utils import coil_compress

n_virtual = 8

kspace_flat = kspace_full.reshape(n_coils, -1)  # (n_coils, n_samples)

kspace_cc, W = coil_compress(kspace_flat, n_virtual)

print(f"\nOriginal coils   : {kspace_flat.shape[0]}")
print(f"Virtual coils    : {kspace_cc.shape[0]}")
print(f"Compression matrix: {W.shape}")

# Reconstruct coil images from compressed data
kspace_cc_full = kspace_cc.reshape(n_virtual, *image_shape)
coil_images_cc = np.fft.fftshift(
    np.fft.ifft2(np.fft.ifftshift(kspace_cc_full, axes=(-2, -1))), axes=(-2, -1)
)

# RSS of compressed coils vs original
rss_orig = np.sqrt((np.abs(coil_images) ** 2).sum(0))
rss_cc = np.sqrt((np.abs(coil_images_cc) ** 2).sum(0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(image, cmap="gray", origin="lower")
axes[0].set_title("Reference")
axes[0].axis("off")
axes[1].imshow(rss_orig, cmap="gray", origin="lower")
axes[1].set_title(f"RSS ({n_coils} coils)")
axes[1].axis("off")
axes[2].imshow(rss_cc, cmap="gray", origin="lower")
axes[2].set_title(f"RSS ({n_virtual} virtual coils, PCA)")
axes[2].axis("off")
plt.suptitle("Coil compression")
plt.tight_layout()
plt.show()

<div class="alert alert-info"><h4>Note</h4><p>Coil compression is lossless when ``n_coils >= original_n_coils`` and
   near-lossless when the retained energy fraction is high (e.g. > 0.99).
   Use a float threshold ``coil_compress(data, 0.99)`` for automatic rank
   selection.</p></div>



## NLINV coil sensitivity estimation

:func:`~pygrog.utils.nlinv_calib` estimates coil sensitivity maps jointly
with the image using the nonlinear-inverse (NLINV) algorithm
(Uecker et al.\ 2008).  It requires a small **fully-sampled ACS centre**
to bootstrap estimation, but no explicit coil sensitivity maps.

The function has two operating modes, both demonstrated below:

* **``cal_width=None``** — full k-space, matching the MATLAB reference:
  NLINV is a *calibrationless image reconstruction* algorithm that jointly
  returns both the image and the sensitivity maps.
* **``cal_width=24``** — crops to a 24×24 central region before solving:
  NLINV becomes a fast *calibration* step that produces low-resolution
  sensitivity maps (zero-padded to full FOV) and a fully-synthesized
  Cartesian k-space patch ready for GRAPPA/GROG kernel training.

The mask mirrors the MATLAB reference: R=2 subsampling in the **readout
(column)** direction with a 16-column fully-sampled ACS centre.



In [ ]:
from pygrog.utils import nlinv_calib

acs = 8         # fully-sampled ACS centre width (columns), as in MATLAB reference
cal_width = 24  # NLINV internal calibration resolution for Step 2

# Cartesian undersampling mask: R=2 readout subsampling + 16-col ACS centre
mask = np.zeros(image_shape, dtype=bool)
mask[:, ::2] = True  # R=2 subsampling on readout (column) direction
cx = image_shape[1] // 2
cy = image_shape[0] // 2
mask[cy - acs // 2 : cy + acs // 2, cx - acs // 2 : cx + acs // 2] = True  # ACS

kspace_us = kspace_full * mask[np.newaxis]

print(f"\nUndersampling factor : {mask.size / mask.sum():.2f}x")
print(f"ACS region           : all rows × {acs} cols")
print(f"Undersampled k-space : {kspace_us.shape}")

ncols_show = min(n_coils // 2, 4)

# Zero-filled RSS image from undersampled data (baseline)
coil_images_us = np.fft.fftshift(
    np.fft.ifft2(np.fft.ifftshift(kspace_us, axes=(-2, -1))), axes=(-2, -1)
)
rss_us = np.sqrt((np.abs(coil_images_us) ** 2).sum(0))

### Step 1 — Calibrationless image reconstruction (``cal_width=None``)
With ``cal_width=None`` NLINV operates on the full k-space matrix, exactly
as in the original paper.  Three rows compare:

* **Row 1** — Reference image and ground-truth sensitivity maps.
* **Row 2** — RSS of the zero-filled k-space (aliased baseline) and the
  NLINV-estimated sensitivity maps (interpolated to full FOV).
* **Row 3** — NLINV-reconstructed image (aliasing removed) with the same maps.



In [ ]:
# cal_width=None → full k-space, no cropping, matching MATLAB reference
smaps_full, _, image_full = nlinv_calib(
    kspace_us,
    cal_width=None,
    ndim=2,
    mask=mask,
    ret_cal=True,
    ret_image=True,
)

smaps_full_np = (
    smaps_full.numpy() if isinstance(smaps_full, torch.Tensor) else smaps_full
)
image_full_np = (
    image_full.numpy() if isinstance(image_full, torch.Tensor) else image_full
)

print(f"\n[Step 1] Smaps shape     : {smaps_full_np.shape}")
print(f"[Step 1] NLINV image shape: {image_full_np.shape}")

In [ ]:
fig, axes = plt.subplots(3, ncols_show + 1, figsize=(3 * (ncols_show + 1), 8))

axes[0, 0].imshow(image, cmap="gray", origin="lower")
axes[0, 0].set_title("Reference")
axes[0, 0].axis("off")
for i in range(ncols_show):
    axes[0, i + 1].imshow(np.abs(smaps_gt[i]), cmap="magma", origin="lower", vmin=0)
    axes[0, i + 1].set_title(f"GT smap {i + 1}")
    axes[0, i + 1].axis("off")

axes[1, 0].imshow(rss_us, cmap="gray", origin="lower")
axes[1, 0].set_title("RSS (zero-filled)")
axes[1, 0].axis("off")
for i in range(ncols_show):
    axes[1, i + 1].imshow(
        np.abs(smaps_full_np[i]), cmap="magma", origin="lower", vmin=0
    )
    axes[1, i + 1].set_title(f"NLINV smap {i + 1}")
    axes[1, i + 1].axis("off")

axes[2, 0].imshow(np.abs(image_full_np), cmap="gray", origin="lower")
axes[2, 0].set_title("NLINV image")
axes[2, 0].axis("off")
for i in range(ncols_show):
    axes[2, i + 1].imshow(
        np.abs(smaps_full_np[i]), cmap="magma", origin="lower", vmin=0
    )
    axes[2, i + 1].set_title(f"NLINV smap {i + 1}")
    axes[2, i + 1].axis("off")

plt.suptitle(
    f"NLINV — calibrationless image reconstruction  ({image_shape[0]}×{image_shape[1]})"
)
plt.tight_layout()
plt.show()

### Step 2 — Calibration mode (``cal_width=24``)
With ``cal_width=24`` NLINV crops to the central 24×24 k-space region
before solving.  The result is a fast, low-resolution calibration that
returns sensitivity maps zero-padded back to the full FOV and a fully
synthesized 24×24 Cartesian calibration k-space.  The latter is exactly
what GRAPPA and GROG kernel training require.

The panel shows the low-resolution image and smaps (what NLINV actually
solved at 24×24), followed by a log-magnitude comparison of the
zero-filled vs NLINV-synthesized central k-space patch.



In [ ]:
smaps_cal, grappa_train, image_cal = nlinv_calib(
    kspace_us,
    cal_width=cal_width,
    ndim=2,
    mask=mask,
    ret_cal=True,
    ret_image=True,
)

smaps_cal_np = (
    smaps_cal.numpy() if isinstance(smaps_cal, torch.Tensor) else smaps_cal
)
grappa_train_np = (
    grappa_train.numpy() if isinstance(grappa_train, torch.Tensor) else grappa_train
)
image_cal_np = (
    image_cal.numpy() if isinstance(image_cal, torch.Tensor) else image_cal
)

print(f"\n[Step 2] Smaps shape       : {smaps_cal_np.shape}")
print(f"[Step 2] Low-res image shape: {image_cal_np.shape}")
print(f"[Step 2] Cal k-space shape  : {grappa_train_np.shape}")

In [ ]:
# Low-res image and smaps at cal_width resolution
fig, axes = plt.subplots(1, ncols_show + 1, figsize=(3 * (ncols_show + 1), 3))

axes[0].imshow(np.abs(image_cal_np), cmap="gray", origin="lower")
axes[0].set_title(f"NLINV image ({cal_width}×{cal_width})")
axes[0].axis("off")
for i in range(ncols_show):
    axes[i + 1].imshow(np.abs(smaps_cal_np[i]), cmap="magma", origin="lower", vmin=0)
    axes[i + 1].set_title(f"NLINV smap {i + 1}")
    axes[i + 1].axis("off")

plt.suptitle(f"NLINV calibration mode — low-res result  ({cal_width}×{cal_width})")
plt.tight_layout()
plt.show()

In [ ]:
# Zero-filled vs synthesized central k-space patch
acr_zf = kspace_us[
    :,
    cy - cal_width // 2 : cy + cal_width // 2,
    cx - cal_width // 2 : cx + cal_width // 2,
]

fig, axes = plt.subplots(2, ncols_show, figsize=(3 * ncols_show, 5.5))
for i in range(ncols_show):
    axes[0, i].imshow(
        np.log1p(np.abs(acr_zf[i])), cmap="inferno", origin="lower"
    )
    axes[0, i].set_title(f"Zero-filled — coil {i + 1}")
    axes[0, i].axis("off")
    axes[1, i].imshow(
        np.log1p(np.abs(grappa_train_np[i])), cmap="inferno", origin="lower"
    )
    axes[1, i].set_title(f"Synthesized — coil {i + 1}")
    axes[1, i].axis("off")

plt.suptitle(
    f"Calibration k-space ({cal_width}×{cal_width})  —  "
    f"zero-filled (top) vs NLINV-synthesized (bottom)"
)
plt.tight_layout()
plt.show()

<div class="alert alert-info"><h4>Note</h4><p>In calibration mode (integer ``cal_width``) the sensitivity maps are
   estimated at low resolution and zero-padded to the full FOV — they are
   smoother but less accurate near the FOV boundary.  Use ``cal_width=None``
   when full image reconstruction quality matters; use an integer
   ``cal_width`` when only the synthesized k-space patch and fast coil
   estimates are needed (e.g.\ as input to GROG/GRAPPA kernel training).</p></div>



In [ ]:
plt.show()

## Multi-slice batched NLINV calibration

:func:`~pygrog.utils.nlinv_calib` accepts a leading batch axis and runs
the calibration once per batch element.  Sensitivity maps and image
reconstructions are returned per-slice; the synthesized GRAPPA training
k-space can optionally be averaged across the batch with
``train_reduce='mean'`` to produce a single shared kernel.



In [ ]:
batch_slices = np.stack([kspace_us, kspace_us[:, ::-1, :]], axis=0)
print(f"\n[Multi-slice] batched k-space shape : {batch_slices.shape}")

# Per-slice smaps + shared (mean-reduced) GRAPPA training k-space.
smaps_b, train_b_mean, image_b = nlinv_calib(
    batch_slices,
    cal_width=cal_width,
    ndim=2,
    ret_cal=True,
    ret_image=True,
    train_reduce="mean",
)
print(f"[Multi-slice] smaps shape           : {tuple(smaps_b.shape)}")
print(f"[Multi-slice] image shape           : {tuple(image_b.shape)}")
print(f"[Multi-slice] mean-train shape      : {tuple(train_b_mean.shape)}")

fig, axes = plt.subplots(2, ncols_show, figsize=(3 * ncols_show, 5.5))
for i in range(ncols_show):
    axes[0, i].imshow(np.abs(smaps_b[0, i]), cmap="magma", origin="lower", vmin=0)
    axes[0, i].set_title(f"slice 0 — smap {i + 1}")
    axes[0, i].axis("off")
    axes[1, i].imshow(np.abs(smaps_b[1, i]), cmap="magma", origin="lower", vmin=0)
    axes[1, i].set_title(f"slice 1 — smap {i + 1}")
    axes[1, i].axis("off")

plt.suptitle("Batched NLINV — per-slice smaps (shared GRAPPA training kernel)")
plt.tight_layout()
plt.show()